# Resume Embedding MS MARCO — GPU (3rd Account)

**Pre-requisites (already done):**
1. Original account shared `msmarco-8.8M-minilm-384d.hdf5` with this account (Editor access).
2. This account added a shortcut to My Drive.
3. Runtime is set to T4 GPU.

**What this notebook does:**
1. Copies the shared file to this account's OWN Drive (shortcuts don't support writes).
2. Embeds the remaining ~2.8M passages on GPU.
3. Writes + flushes each chunk. Checkpoints for crash recovery.
4. Final file lives on this account's Drive — share it back when done.

**Time: ~1.5–2 hours on T4 GPU.**

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Install dependencies
!pip install -q sentence-transformers datasets h5py

In [ ]:
# Cell 3: Find the shared HDF5 file and copy it to this account's own Drive
import os
import shutil

FILENAME = "msmarco-8.8M-minilm-384d.hdf5"
# This is the file we will ACTUALLY work on (owned by THIS account, not a shortcut)
OUTPUT_FILE = f"/content/drive/MyDrive/msmarco_LOCAL_COPY.hdf5"

# --- Step A: Check if we already have our own copy from a previous run ---
if os.path.exists(OUTPUT_FILE):
    size_gb = os.path.getsize(OUTPUT_FILE) / (1024**3)
    print(f"✓ Local copy already exists: {OUTPUT_FILE}")
    print(f"  Size: {size_gb:.2f} GB")
    print("  Skipping copy step (using existing file).")
else:
    # --- Step B: Find the shared/shortcut file ---
    SHARED_FILE = None
    SEARCH_PATHS = [
        f"/content/drive/MyDrive/{FILENAME}",
        f"/content/drive/Shareddrives/{FILENAME}",
    ]
    for p in SEARCH_PATHS:
        if os.path.exists(p):
            SHARED_FILE = p
            break
    
    if SHARED_FILE is None:
        print(f"Searching entire Drive for {FILENAME}...")
        for root, dirs, files in os.walk('/content/drive'):
            if FILENAME in files:
                SHARED_FILE = os.path.join(root, FILENAME)
                break
    
    if SHARED_FILE is None:
        print("ERROR: Cannot find the shared HDF5 file!")
        print("Run this to debug: !find /content/drive -name '*.hdf5' -type f 2>/dev/null")
        raise FileNotFoundError(FILENAME)
    
    print(f"Found shared file: {SHARED_FILE}")
    src_size = os.path.getsize(SHARED_FILE) / (1024**3)
    print(f"  Size: {src_size:.2f} GB")
    
    # --- Step C: Copy to this account's own Drive ---
    print(f"\nCopying to {OUTPUT_FILE}...")
    print("(This takes a few minutes — it's a ~13 GB file)")
    shutil.copy2(SHARED_FILE, OUTPUT_FILE)
    dst_size = os.path.getsize(OUTPUT_FILE) / (1024**3)
    print(f"✓ Copy complete! Size: {dst_size:.2f} GB")

# --- Verify write access on our own copy ---
try:
    with open(OUTPUT_FILE, 'r+b') as f:
        pass
    print("✓ Write access confirmed.")
except PermissionError:
    raise PermissionError(f"Cannot write to {OUTPUT_FILE}!")

In [ ]:
# Cell 4: Imports and Config
import os
import json
import time
import h5py
import numpy as np
import gc
import torch
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from tqdm.auto import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CHECKPOINT_FILE = "/content/drive/MyDrive/msmarco_embed_checkpoint.json"
EMBEDDING_DIM = 384
BATCH_SIZE = 1024
CHUNK_SIZE = 100_000
HARD_RESUME_FROM = 6_000_000

print(f"Output:     {OUTPUT_FILE}")
print(f"Checkpoint: {CHECKPOINT_FILE}")
print(f"Batch:      {BATCH_SIZE}")
print(f"Chunk:      {CHUNK_SIZE:,}")
print(f"Resume:     {HARD_RESUME_FROM:,}")

In [ ]:
# Cell 5: Load model on GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("  ⚠️  No GPU — will run on CPU (slower but works)")

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
model.max_seq_length = 256
print(f"Model loaded on {device}.")

In [ ]:
# Cell 6: Load the dataset
print("Loading BeIR MS MARCO corpus...")
dataset = load_dataset("BeIR/msmarco", "corpus", split="corpus")
TOTAL_N = len(dataset)
print(f"Total:     {TOTAL_N:,}")
print(f"Done:      {HARD_RESUME_FROM:,}")
print(f"Remaining: {TOTAL_N - HARD_RESUME_FROM:,}")

In [ ]:
# Cell 7: Verify the HDF5 file
print("Verifying HDF5 file...")
with h5py.File(OUTPUT_FILE, 'r') as f:
    assert 'train' in f, "No 'train' dataset!"
    print(f"  Shape: {f['train'].shape}")
    print(f"  Dtype: {f['train'].dtype}")
    for idx in [0, 1_000_000, 3_000_000, 5_999_999]:
        norm = np.linalg.norm(f['train'][idx])
        print(f"  Row {idx:>10,}: norm={norm:.4f} {'✓' if norm > 0.1 else '✗ ZERO'}")
    # Check row right after 6M
    norm_6m = np.linalg.norm(f['train'][6_000_000])
    print(f"  Row  6,000,000: norm={norm_6m:.4f} {'(has data)' if norm_6m > 0.1 else '(empty — expected)'}")
print("✓ Verified.")

In [ ]:
# Cell 8: Checkpoint helpers

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as ckf:
                return json.load(ckf).get('last_completed_index', HARD_RESUME_FROM)
        except:
            pass
    return HARD_RESUME_FROM

def save_checkpoint(idx):
    with open(CHECKPOINT_FILE, 'w') as ckf:
        json.dump({'last_completed_index': idx, 'ts': time.strftime('%Y-%m-%d %H:%M:%S')}, ckf)

START_INDEX = load_checkpoint()
print(f"Resuming from: {START_INDEX:,}")
print(f"To embed:      {TOTAL_N - START_INDEX:,}")

In [ ]:
# Cell 9: Prepare texts (only remaining passages)

def get_text(example):
    title = example.get('title', '') or ''
    text = example.get('text', '') or ''
    combined = (title + ' ' + text).strip() if title else text.strip()
    return combined if combined else 'empty'

print(f"Preparing texts [{START_INDEX:,} → {TOTAL_N:,}]...")
remaining_dataset = dataset.select(range(START_INDEX, TOTAL_N))
texts = [get_text(ex) for ex in tqdm(remaining_dataset, desc="Preparing")]
print(f"Prepared {len(texts):,} texts.")

del remaining_dataset, dataset
gc.collect()
print("Dataset freed from memory.")

In [ ]:
# Cell 10: MAIN EMBEDDING LOOP

total_to_embed = len(texts)
num_chunks = (total_to_embed + CHUNK_SIZE - 1) // CHUNK_SIZE
t0 = time.time()

print(f"Embedding {total_to_embed:,} passages in {num_chunks} chunks")
print(f"Batch size: {BATCH_SIZE} | Device: {device}")
print("=" * 70)

for chunk_idx in range(num_chunks):
    local_start = chunk_idx * CHUNK_SIZE
    local_end = min(local_start + CHUNK_SIZE, total_to_embed)
    global_start = START_INDEX + local_start
    global_end = START_INDEX + local_end

    # Skip already-checkpointed chunks
    if global_end <= load_checkpoint():
        print(f"Chunk {chunk_idx+1}/{num_chunks} (rows {global_start:,}–{global_end:,}) — skipped.")
        continue

    chunk_texts = texts[local_start:local_end]
    chunk_len = len(chunk_texts)

    print(f"\n{'=' * 70}")
    print(f"Chunk {chunk_idx+1}/{num_chunks}: rows {global_start:,} → {global_end:,} ({chunk_len:,})")
    chunk_t0 = time.time()

    # --- Encode with error handling ---
    all_embeddings = np.zeros((chunk_len, EMBEDDING_DIM), dtype=np.float32)
    num_failed = 0

    for batch_start in tqdm(range(0, chunk_len, BATCH_SIZE),
                            desc="Encoding",
                            total=(chunk_len + BATCH_SIZE - 1) // BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, chunk_len)
        batch_texts = chunk_texts[batch_start:batch_end]

        try:
            batch_emb = model.encode(
                batch_texts,
                batch_size=BATCH_SIZE,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=False
            )
            bad = ~np.isfinite(batch_emb).all(axis=1)
            if bad.any():
                batch_emb[bad] = 0.0
                num_failed += int(bad.sum())
            all_embeddings[batch_start:batch_end] = batch_emb
            del batch_emb
        except Exception as e:
            print(f"  ⚠️  Batch failed ({e}). One-by-one fallback...")
            for j, txt in enumerate(batch_texts):
                try:
                    emb = model.encode([txt], batch_size=1, show_progress_bar=False, convert_to_numpy=True)
                    if np.isfinite(emb).all():
                        all_embeddings[batch_start + j] = emb[0]
                    else:
                        num_failed += 1
                except:
                    num_failed += 1

        if device == 'cuda' and batch_start % (BATCH_SIZE * 10) == 0:
            torch.cuda.empty_cache()

    # --- Write to HDF5 (open, write, flush, close — every chunk) ---
    print(f"  Writing to HDF5...")
    wt0 = time.time()
    with h5py.File(OUTPUT_FILE, 'a') as hf:
        hf['train'][global_start:global_end] = all_embeddings
        hf.flush()
    print(f"  Written + flushed in {time.time() - wt0:.1f}s")

    # --- Verify by re-opening the file ---
    with h5py.File(OUTPUT_FILE, 'r') as hf:
        mid = chunk_len // 2
        disk_row = hf['train'][global_start + mid]
        match = np.allclose(disk_row, all_embeddings[mid], atol=1e-6)
        print(f"  Verify row {global_start + mid:,}: {'✓ OK' if match else '✗ MISMATCH!'}")
        if not match:
            print("  ✗ DATA DID NOT PERSIST. Stopping to avoid wasting GPU time.")
            print("    This account may not own the file. Check OUTPUT_FILE path.")
            break

    # --- Checkpoint ---
    save_checkpoint(global_end)

    # --- Progress ---
    chunk_time = time.time() - chunk_t0
    done = global_end - START_INDEX
    elapsed = time.time() - t0
    rate = done / elapsed if elapsed > 0 else 1
    eta = (TOTAL_N - global_end) / rate if rate > 0 else 0

    print(f"  Time: {chunk_time / 60:.1f} min | Failed: {num_failed}")
    print(f"  Progress: {global_end:,}/{TOTAL_N:,} ({global_end / TOTAL_N * 100:.1f}%)")
    print(f"  ETA: {eta / 60:.0f} min ({eta / 3600:.1f} hr)")

    del all_embeddings, chunk_texts
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

total_time = time.time() - t0
print(f"\n{'=' * 70}")
print(f"✓ DONE! Total: {total_time / 3600:.1f} hr ({total_time / 60:.0f} min)")
print(f"  File: {OUTPUT_FILE}")
print(f"{'=' * 70}")

In [ ]:
# Cell 11: Final Verification

import h5py, numpy as np, os

print("Final verification of the complete file...")
print("=" * 70)

with h5py.File(OUTPUT_FILE, 'r') as f:
    dset = f['train']
    print(f"Shape: {dset.shape}")
    print(f"Size:  {os.path.getsize(OUTPUT_FILE) / (1024**3):.2f} GB")

    check_rows = list(range(0, dset.shape[0], 1_000_000)) + [dset.shape[0] - 1]
    print(f"\nSpot-checking {len(check_rows)} rows:")
    for idx in check_rows:
        norm = np.linalg.norm(dset[idx])
        print(f"  Row {idx:>10,}: norm={norm:.4f} {'✓' if norm > 0.1 else '✗ ZERO'}")

    print(f"\nSampling 10,000 random rows...")
    sample = np.sort(np.random.choice(dset.shape[0], 10000, replace=False))
    norms = np.linalg.norm(dset[sample], axis=1)
    pct_zero = (norms < 0.01).sum() / len(norms) * 100
    print(f"  Zero-vector rate: {pct_zero:.2f}%")
    print(f"  {'✓ GOOD' if pct_zero < 1 else '⚠️ Some chunks may be missing — re-run notebook'}")

# Cleanup
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print(f"\nCheckpoint cleaned up.")

print(f"\n{'=' * 70}")
print(f"✓ File ready: {OUTPUT_FILE}")
print(f"Share this file back to your main account from this account's Drive.")
print(f"{'=' * 70}")